# FinanceBench — exploring the real-filings corpus

Run block by block to understand `src/policy_copilot/financebench.py`: turning 8 real 10-K/10-Q filings into a searchable corpus.

**How to run:** in VS Code pick the `.venv` kernel, or in a terminal `uv run jupyter lab`.

> Note: the first `build_corpus` call downloads + parses the PDFs (~1–2 min); after that it is cached and instant.

## 1. Load the 150 questions

In [ ]:
from policy_copilot.financebench import (
    build_corpus,
    load_rows,
    questions_for,
    select_documents,
)

rows = load_rows()
len(rows)

## 2. Look at one real question

Each row has: `question`, `answer`, `doc_name` (source filing), `evidence` (gold supporting text).

In [ ]:
r = rows[0]
print("Company :", r["company"])
print("Source  :", r["doc_name"])
print("Question:", r["question"])
print("Answer  :", r["answer"])
print("Evidence:", r["evidence"][0]["evidence_text"][:200], "...")

## 3. Select the lean subset (8 most-questioned documents)

In [ ]:
doc_names = select_documents(rows, n=8)
doc_names

In [ ]:
questions = questions_for(rows, doc_names)
print(f"These 8 documents cover {len(questions)} questions")

## 4. Build the corpus (download + parse, cached)

In [ ]:
corpus = build_corpus(rows, doc_names)
for f in corpus:
    print(f"{f.doc_id:30} {f.n_pages:>4} pages  {len(f.text):>9,} chars")

## 5. Needle in a haystack — why retrieval is a real problem

Take one question and see how tiny its gold evidence is relative to the whole filing.

In [ ]:
doc = doc_names[0]
q = next(r for r in rows if r["doc_name"] == doc)
filing = next(f for f in corpus if f.doc_id == doc)
evidence = q["evidence"][0]["evidence_text"] if q["evidence"] else ""
pct = len(evidence) / len(filing.text) * 100

print("Question:", q["question"])
print("Answer  :", q["answer"])
print(f"\nWhole document : {filing.n_pages} pages / {len(filing.text):,} chars")
print(f"Gold evidence  : {len(evidence):,} chars  ({pct:.3f}% of the doc)")
print("\nEvidence preview:\n", evidence[:300])

## 6. Peek at the parsed Markdown

pymupdf4llm converts each PDF to Markdown, preserving heading structure so we can chunk it later.

In [ ]:
print(filing.text[:1500])